<a href="https://colab.research.google.com/github/fce-laboratoriotp/tp-labo/blob/main/notebook_prueba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prueba de Commit Grupal - Laboratorio de Métodos Cuantitativos
Este es un archivo de prueba para verificar la conexión con el repositorio de GitHub y asegurar que los commits de todas las integrantes queden registrados correctamente.

In [9]:
import pandas as pd


hola soy miguelina

hola soy aldi

Hola, soy Kay


hola soy mica

# **Análisis Exploratorio y Detección de Variables**

En esta etapa inicial examinamos individualmente las fuentes de datos disponibles para evaluar su estructura, identificar qué información aporta cada una y detectar posibles vacíos (como la ausencia de desgloses demográficos específicos en los agregados generales).

In [10]:
import pandas as pd

# 1. Cargamos la base general del SNIC a nivel pais
df_snic = pd.read_csv("snic-pais.csv", sep=";", encoding="latin1", on_bad_lines="skip")

# 2. Miro el tamaño (cuántas filas y columnas tiene)
print("Dimensiones de la tabla:", df_snic.shape)

# 3. Reviso las primeras filas para entender cómo vienen los datos
display(df_snic.head(3))

# 4. Reviso los tipos de datos de cada columna (para ver si los números son números o texto)
print("\nTipos de datos:")
print(df_snic.dtypes)

# 5. Chequeo si hay valores nulos o celdas vacías
print("\nValores nulos:")
print(df_snic.isnull().sum())

Dimensiones de la tabla: (1112, 12)


,anio,codigo_delito_snic_id,codigo_delito_snic_nombre,cantidad_hechos,cantidad_victimas,cantidad_victimas_masc,cantidad_victimas_fem,cantidad_victimas_sd,tasa_hechos,tasa_victimas,tasa_victimas_masc,tasa_victimas_fem
0,2000,1,Homicidios dolosos,2653,2805.0,2167.0,369.0,269.0,"7,2270207","7,6410828","12,048265","1,9707898"
1,2000,2,Homicidios dolosos en grado de tentativa,884,942.0,641.0,135.0,177.0,"2,4080989","2,5660963","3,5638845","0,72102064"
2,2000,3,Muertes en accidentes viales,3686,4297.0,2932.0,927.0,438.0,"10,041009","11,705431","16,301575","4,9510083"



Tipos de datos:
anio                           int64
codigo_delito_snic_id         object
codigo_delito_snic_nombre     object
cantidad_hechos                int64
cantidad_victimas            float64
cantidad_victimas_masc       float64
cantidad_victimas_fem        float64
cantidad_victimas_sd         float64
tasa_hechos                   object
tasa_victimas                 object
tasa_victimas_masc            object
tasa_victimas_fem             object
dtype: object

Valores nulos:
anio                           0
codigo_delito_snic_id          0
codigo_delito_snic_nombre      0
cantidad_hechos                0
cantidad_victimas            732
cantidad_victimas_masc       732
cantidad_victimas_fem        732
cantidad_victimas_sd         732
tasa_hechos                    0
tasa_victimas                732
tasa_victimas_masc           732
tasa_victimas_fem            732
dtype: int64


In [11]:
# Filtramos los nulos y vemos cuántos son de suicidios
nulos_df = df_snic[df_snic["cantidad_victimas"].isnull()]

# Contamos cuántos pertenecen a la categoría de suicidios
suicidios_nulos = nulos_df[nulos_df["codigo_delito_snic_nombre"].astype(str).str.contains("Suicidios", case=False, na=False)]

print(f"Total de filas con nulos en todo el dataset: {len(nulos_df)}")
print(f"De esos nulos, ¿cuántos corresponden a suicidios?: {len(suicidios_nulos)}")

if len(suicidios_nulos) == 0:
    print("Ninguno de los nulos afecta a los suicidios. Están en otros delitos.")
else:
    print("hay registros nulos en suicidios. Vamos a ver en qué años ocurren:")
    display(suicidios_nulos[["anio", "codigo_delito_snic_nombre"]])

Total de filas con nulos en todo el dataset: 732
De esos nulos, ¿cuántos corresponden a suicidios?: 0
Ninguno de los nulos afecta a los suicidios. Están en otros delitos.


### **Extracción y acotamiento del universo de estudio**

Dado que la base general del SNIC contiene múltiples tipologías delictivas, el siguiente paso metodológico consiste en extraer exclusivamente los registros asociados al delito de **Suicidios** y acotar el horizonte temporal al quinquenio **2020-2024**, en consonancia con los objetivos del trabajo.

In [12]:
# Seleccionamos los registros de suicidios para el periodo de estudio
df_snic_suicidios = df_snic[
    (df_snic["codigo_delito_snic_nombre"].astype(str).str.contains("Suicidios", case=False, na=False)) &
    (df_snic["anio"].between(2020, 2024))
].copy()

print(f"Cantidad de observaciones seleccionadas para el análisis: {len(df_snic_suicidios)}")
display(df_snic_suicidios.head(3))

Cantidad de observaciones seleccionadas para el análisis: 5


,anio,codigo_delito_snic_id,codigo_delito_snic_nombre,cantidad_hechos,cantidad_victimas,cantidad_victimas_masc,cantidad_victimas_fem,cantidad_victimas_sd,tasa_hechos,tasa_victimas,tasa_victimas_masc,tasa_victimas_fem
774,2020,31,Suicidios (consumados),3261,3262.0,2611.0,643.0,8.0,"7,82694337771705","7,82934354434622","12,8225470960958","3,0186085716747"
834,2021,31,Suicidios (consumados),3648,3648.0,2882.0,754.0,12.0,"8,66303898742753","8,66303898742753","13,9974258724107","3,50364681307078"
894,2022,31,Suicidios (consumados),3959,3959.0,3063.0,859.0,37.0,"9,1652972352889","9,1652972352889","13,9069984574719","4,05750183910168"


### **Inspección de la Dimensión Territorial (Desglose Provincial)**

La primera base analizada (`df_snic`) nos proporciona la tendencia macro a nivel nacional, pero **carece del desglose por jurisdicción o provincia**, lo cual impide evaluar las diferencias geográficas y la incidencia relativa del fenómeno según la población de cada territorio.

Para resolver esta limitación, procedemos a cargar e inspeccionar la base provincial oficial (`snic-provincias.csv`). Verificaremos su estructura, las variables disponibles y aplicaremos los mismos criterios de auditoría e inclusión (filtrado del delito de **Suicidios** y ventana temporal **2020-2024**) para asegurar la consistencia del análisis.

In [13]:
import pandas as pd

# 1. Carga de la base de datos a nivel provincial
df_prov = pd.read_csv("snic-provincias.csv", sep=";", encoding="latin1", on_bad_lines="skip")

# 2. Inspección estructural de la base provincial
print("--- 1. Dimensiones de la base provincial ---")
print(f"Filas: {df_prov.shape[0]} | Columnas: {df_prov.shape[1]}")

print("\n--- 2. Columnas disponibles en la base provincial ---")
print(df_prov.columns.tolist())

# 3. Vista previa de los datos
display(df_prov.head(3))

--- 1. Dimensiones de la base provincial ---
Filas: 26528 | Columnas: 14

--- 2. Columnas disponibles en la base provincial ---
['provincia_id', 'provincia_nombre', 'anio', 'codigo_delito_snic_id', 'codigo_delito_snic_nombre', 'cantidad_hechos', 'cantidad_victimas', 'cantidad_victimas_masc', 'cantidad_victimas_fem', 'cantidad_victimas_sd', 'tasa_hechos', 'tasa_victimas', 'tasa_victimas_masc', 'tasa_victimas_fem']


,provincia_id,provincia_nombre,anio,codigo_delito_snic_id,codigo_delito_snic_nombre,cantidad_hechos,cantidad_victimas,cantidad_victimas_masc,cantidad_victimas_fem,cantidad_victimas_sd,tasa_hechos,tasa_victimas,tasa_victimas_masc,tasa_victimas_fem
0,2,Ciudad AutÃ³noma de Buenos Aires,2000,1,Homicidios dolosos,149,158.0,121.0,37.0,0.0,"4,9750013","5,2755051","8,8257008","2,2783601"
1,2,Ciudad AutÃ³noma de Buenos Aires,2000,2,Homicidios dolosos en grado de tentativa,54,58.0,40.0,18.0,0.0,"1,8030207","1,9365778","2,9175868","1,1083914"
2,2,Ciudad AutÃ³noma de Buenos Aires,2000,3,Muertes en accidentes viales,121,126.0,97.0,29.0,0.0,"4,040102","4,2070484","7,0751481","1,7857418"


### **Homogeneización y validación cruzada entre bases**

Para asegurar la robustez del análisis, filtramos la base provincial bajo los mismos criterios (período 2020-2024 y tipología de suicidios). Posteriormente, efectuamos un cotejo cuantitativo entre la sumatoria de los registros provinciales y los totales informados a nivel nacional para verificar la integridad de los datos provistos por el SNIC.

In [14]:
# 1. Filtramos la base provincial por suicidios y periodo 2020-2024
df_prov_suicidios = df_prov[
    (df_prov["codigo_delito_snic_nombre"].astype(str).str.contains("Suicidios", case=False, na=False)) &
    (df_prov["anio"].between(2020, 2024))
].copy()

print(f"Registros en la base provincial filtrada: {len(df_prov_suicidios)}")

# 2. Agrupamos y sumamos a nivel provincial por año
suma_provincias = df_prov_suicidios.groupby("anio")["cantidad_victimas"].sum().reset_index(name="suma_provincias")

# 3. Tomamos los totales de la base nacional
total_nacional = df_snic_suicidios[["anio", "cantidad_victimas"]].rename(columns={"cantidad_victimas": "total_nacional"})

# 4. Cruzamos ambas para comparar
validacion_cruzada = pd.merge(total_nacional, suma_provincias, on="anio")
validacion_cruzada["diferencia"] = validacion_cruzada["total_nacional"] - validacion_cruzada["suma_provincias"]

print("\n--- Resultado de la validación cruzada (Nacional vs Suma Provincial) ---")
display(validacion_cruzada)

Registros en la base provincial filtrada: 120

--- Resultado de la validación cruzada (Nacional vs Suma Provincial) ---


,anio,total_nacional,suma_provincias,diferencia
0,2020,3262.0,3262.0,0.0
1,2021,3648.0,3648.0,0.0
2,2022,3959.0,3959.0,0.0
3,2023,4205.0,4205.0,0.0
4,2024,4249.0,4249.0,0.0


### **Análisis comparativo de esquemas entre fuentes**

A fin de evaluar la compatibilidad estructural de los datos, contrastamos las columnas de la base nacional (`snic-pais.csv`) con las de la base provincial (`snic-provincias.csv`). Este procedimiento permite verificar la presencia de identificadores geográficos en la segunda tabla y asegurar que las variables métricas sean equivalentes para el análisis.

In [15]:
# Convertimos las listas de columnas a conjuntos (sets) para realizar la comparación de diferencias y intersecciones
cols_pais = set(df_snic_suicidios.columns)
cols_prov = set(df_prov_suicidios.columns)

solo_en_pais = cols_pais - cols_prov
solo_en_prov = cols_prov - cols_pais
comunes = cols_pais.intersection(cols_prov)

print(f"Total de columnas en base país: {len(cols_pais)}")
print(f"Total de columnas en base provincial: {len(cols_prov)}")

print("\n--- Columnas presentes en ambas bases ---")
print(sorted(list(comunes)))

print("\n--- Columnas exclusivas de la base país ---")
print(sorted(list(solo_en_pais)))

print("\n--- Columnas exclusivas de la base provincial ---")
print(sorted(list(solo_en_prov)))

Total de columnas en base país: 12
Total de columnas en base provincial: 14

--- Columnas presentes en ambas bases ---
['anio', 'cantidad_hechos', 'cantidad_victimas', 'cantidad_victimas_fem', 'cantidad_victimas_masc', 'cantidad_victimas_sd', 'codigo_delito_snic_id', 'codigo_delito_snic_nombre', 'tasa_hechos', 'tasa_victimas', 'tasa_victimas_fem', 'tasa_victimas_masc']

--- Columnas exclusivas de la base país ---
[]

--- Columnas exclusivas de la base provincial ---
['provincia_id', 'provincia_nombre']


### **Selección de la base de trabajo y apertura a microdatos**

Tras haber auditado y comparado el esquema de ambas fuentes, se determinó utilizar como base principal la tabla provincial (`snic-provincias.csv`). Al incluir la totalidad de las variables métricas de la base nacional más los identificadores geográficos (`provincia_id` y `provincia_nombre`), resulta un dataset más completo e integral para abordar tanto el análisis macro como el territorial.

Sin embargo, para dar cumplimiento a los objetivos del trabajo práctico, evaluaremos la factibilidad de incorporar una tercera fuente: la base de **microdatos** (`SAT-SS-BU_2017-2024.csv`). Procederemos a cargarla y compararla estructuralmente con la base provincial para determinar qué variables específicas aporta (por ejemplo, la desagregación por tramos etarios) y verificar su compatibilidad y cobertura.

In [16]:
# 1. Cargamos la base de microdatos
df_micro = pd.read_csv("SAT-SS-BU_2017-2024.csv", sep=",", encoding="latin1", on_bad_lines="skip")

# 2. Inspeccionamos dimensiones y columnas para poder comparar
print("--- Dimensiones de la base de Microdatos ---")
print(f"Filas: {df_micro.shape[0]} | Columnas: {df_micro.shape[1]}")

print("\n--- Columnas disponibles en Microdatos ---")
print(df_micro.columns.tolist())

# 3. Vista preliminar de los registros
display(df_micro.head(3))

--- Dimensiones de la base de Microdatos ---
Filas: 30177 | Columnas: 27

--- Columnas disponibles en Microdatos ---
['id_hecho', 'tipo_persona_id', 'federal', 'provincia_id', 'provincia_nombre', 'departamento_id', 'departamento_nombre', 'localidad_id', 'localidad_nombre', 'anio', 'mes', 'fecha_hecho', 'hora_hecho', 'tipo_lugar', 'tipo_lugar_ampliado', 'tipo_lugar_otro', 'modalidad', 'modalidad_ampliado', 'modalidad_otro', 'motivo_origen_registro', 'motivo_origen_registro_otro', 'suicida_sexo', 'suicida_tr_edad', 'suicida_18_aÃ±os_o_mas', 'suicida_clase', 'suicida_clase_otro', 'suicida_identidad_genero']


,id_hecho,tipo_persona_id,federal,provincia_id,provincia_nombre,departamento_id,departamento_nombre,localidad_id,localidad_nombre,anio,...,modalidad_ampliado,modalidad_otro,motivo_origen_registro,motivo_origen_registro_otro,suicida_sexo,suicida_tr_edad,suicida_18_aÃ±os_o_mas,suicida_clase,suicida_clase_otro,suicida_identidad_genero
0,8529,Suicida idRegistro 20712,No,42,La Pampa,42002.0,Norte (General Pico),105030,General Pico,2017,...,Ahorcamiento,NaN,IntervenciÃ³n policial,NaN,Masculino,20-24,Si,Sin determinaciÃ³n,NaN,Sin determinar
1,8530,Suicida idRegistro 20713,No,42,La Pampa,42003.0,Sur (General Acha),70030,GuatrachÃ©,2017,...,Arma de fuego,NaN,IntervenciÃ³n policial,NaN,Masculino,55-59,Si,Sin determinaciÃ³n,NaN,Sin determinar
2,8569,Suicida idRegistro 20752,No,78,Santa Cruz,78021.0,GÃ¼er Aike,60,28 de Noviembre,2017,...,Ahorcamiento,NaN,Denuncia particular,NaN,Masculino,15-19,No,Sin determinaciÃ³n,NaN,Sin determinar


In [21]:
# 1. Filtramos los microdatos solo para 2020-2024
df_micro_2020 = df_micro[df_micro["anio"].between(2020, 2024)]

# 2. Comparamos los totales de ambos
print("Víctimas en microdatos (2020-2024):", len(df_micro_2020))
print("Víctimas en base provincial (2020-2024):", df_prov_suicidios["cantidad_victimas"].sum())

Víctimas en microdatos (2020-2024): 19323
Víctimas en base provincial (2020-2024): 19323.0


### **Validación y complementariedad de la base de microdatos**

La inspección de la base de microdatos (`SAT-SS-BU_2017-2024.csv`) evidencia un total de 30.177 registros y 27 columnas. A diferencia de la tabla provincial, esta base opera a nivel de evento individual y aporta variables demográficas clave que no estaban disponibles en los agregados oficiales, tales como **tramos de edad** (`suicida_tr_edad`), **sexo** (`suicida_sexo`) e **identidad de género** (`suicida_identidad_genero`).

De este modo, se establece una estrategia de fuentes combinadas:
1. **Base Provincial (`snic-provincias.csv`):** Se utilizará para el análisis macroeconómico, la evolución temporal y el cálculo de tasas oficiales por jurisdicción.
2. **Base de Microdatos (`SAT-SS-BU_2017-2024.csv`):** Se empleará para el análisis demográfico multidimensional (recorte 2020-2024), abordando la distribución etaria, la brecha por género, el cruce bidimensional entre edad y sexo, y la evaluación de la calidad de registro en identidad de género.